# Tutorial 4: LSTM and ANN

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_absolute_error, r2_score
import os
import copy

# Configure tqdm for pandas operations
tqdm.pandas()

flat_filepath = os.path.join('dataset', 'data', 'FlatFiles', 'NGAsub_MegaFlatfile_RotD50_050_R211022_public.xlsx')
OUTPUT_DIR    = os.path.join('assignments', '4')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✓ PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Device: {device}")
print(f"  flat_filepath : {flat_filepath}")
print(f"  OUTPUT_DIR    : {OUTPUT_DIR}")


✓ PyTorch version: 2.6.0+cu124
✓ Device: cuda
  flat_filepath : dataset/data/FlatFiles/NGAsub_MegaFlatfile_RotD50_050_R211022_public.xlsx
  OUTPUT_DIR    : assignments/4


In [2]:
# Load data efficiently
# Helper function to convert period column name to float
def period_to_float(col_name):
    """Convert column name like T0pt010S to 0.010"""
    period_str = col_name[1:-1].replace('pt', '.')
    return float(period_str)

# Get all column names first
print("Reading column names")
df_temp  = pd.read_excel(flat_filepath, engine='calamine', nrows=0)
all_cols = df_temp.columns.tolist()

# Find period columns <= 4.0 seconds only
period_cols             = [col for col in all_cols if col.startswith('T') and col.endswith('S')]
selected_period_cols    = [col for col in period_cols if period_to_float(col) <= 4.0]
selected_period_cols_sorted = sorted(selected_period_cols, key=period_to_float)

# Define columns to load
selected_columns = [
    'Earthquake_Magnitude',
    'Rjb_km',
    'Vs30_Selected_for_Analysis_m_s',
    'Fault_Type',
    'PGA_g',
    'PGV_cm_sec'
] + selected_period_cols_sorted

# Define dtypes for faster loading (avoid type inference)
dtype_dict = {
    'Earthquake_Magnitude':             'float32',
    'Rjb_km':                           'float32',
    'Vs30_Selected_for_Analysis_m_s':   'float32',
    'Fault_Type':                       'int8',
    'PGA_g':                            'float32',
    'PGV_cm_sec':                       'float32'
}
for col in selected_period_cols_sorted:
    dtype_dict[col] = 'float32'

print(f"Loading {len(selected_columns)} columns (6 base features + {len(selected_period_cols_sorted)} periods)...")
print(f"Period range: {period_to_float(selected_period_cols_sorted[0]):.3f}s to {period_to_float(selected_period_cols_sorted[-1]):.3f}s")

df = pd.read_excel(flat_filepath, engine='calamine', usecols=selected_columns, dtype=dtype_dict)

print(f"✓ Loaded data shape: {df.shape}")
print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


Reading column names
Loading 96 columns (6 base features + 90 periods)...
Period range: 0.010s to 4.000s
✓ Loaded data shape: (71340, 96)
  Memory usage: 25.92 MB


In [3]:
# Data Cleaning: Remove outliers and invalid values
print("="*80)
print("DATA CLEANING")
print("="*80)
print(f"\nOriginal shape: {df.shape}")

# Filter 1: Magnitude >= 4
df = df[df['Earthquake_Magnitude'] >= 4]
print(f"After M >= 4 filter: {df.shape}")

# Filter 2: Distance 0-500 km
df = df[(df['Rjb_km'] > 0) & (df['Rjb_km'] <= 500)]
print(f"After distance filter: {df.shape}")

# Filter 3: Remove rows with Vs30 <= 0
df = df[df['Vs30_Selected_for_Analysis_m_s'] > 0]
print(f"After Vs30 > 0 filter: {df.shape}")

# Filter 4: PGA realistic range (0-5g)
df = df[(df['PGA_g'] > 0) & (df['PGA_g'] < 5)]
print(f"After PGA filter: {df.shape}")

# Filter 5: PGV realistic range (0-500 cm/s)
df = df[(df['PGV_cm_sec'] > 0) & (df['PGV_cm_sec'] < 500)]
print(f"After PGV filter: {df.shape}")

# Filter 6: Remove rows with invalid spectral values (with progress bar)
print(f"\nFiltering {len(selected_period_cols_sorted)} spectral acceleration columns...")
for col in tqdm(selected_period_cols_sorted, desc="Filtering Sa columns"):
    df = df[(df[col] > 0) & (df[col] < 5)]

print(f"After spectral value filters: {df.shape}")

# Reset index
df = df.reset_index(drop=True)
print(f"\n✓ Final cleaned data: {len(df):,} earthquakes")
print("="*80)

DATA CLEANING

Original shape: (71340, 96)
After M >= 4 filter: (68473, 96)
After distance filter: (55298, 96)
After Vs30 > 0 filter: (55059, 96)
After PGA filter: (54972, 96)
After PGV filter: (54972, 96)

Filtering 90 spectral acceleration columns...


Filtering Sa columns: 100%|██████████| 90/90 [00:00<00:00, 481.62it/s]

After spectral value filters: (54967, 96)

✓ Final cleaned data: 54,967 earthquakes


In [4]:
df['Rjb_km'].describe()

count    54967.000000
mean       200.558365
std        120.732170
min          0.005944
25%        103.719273
50%        181.756287
75%        283.649216
max        499.997467
Name: Rjb_km, dtype: float64

In [5]:
df['Vs30_Selected_for_Analysis_m_s'].describe()

count    54967.000000
mean       429.677521
std        216.375931
min         53.000000
25%        270.000000
50%        389.600006
75%        527.754242
max       2229.800049
Name: Vs30_Selected_for_Analysis_m_s, dtype: float64

In [6]:
# Data Processing: Create log-transformed features and targets
print("="*80)
print("DATA PREPROCESSING")
print("="*80)

df = df.reset_index(drop=True)

# Create log10-transformed features (will be used as inputs)
print("\nCreating log-transformed input features...")
df['log10_Rjb_km'] = np.log10(df['Rjb_km'])
df['log10_Vs30'] = np.log10(df['Vs30_Selected_for_Analysis_m_s'])
print("  ✓ log10_Rjb_km, ✓ log10_Vs30")

# Create log10-transformed targets (will be used as outputs)
df['log10_PGA_g'] = np.log10(df['PGA_g'])
df['log10_PGV_cm_sec'] = np.log10(df['PGV_cm_sec'])
print("  ✓ log10_PGA_g, ✓ log10_PGV_cm_sec")

# Add log10 for all spectral acceleration periods (with progress bar)
print(f"\nCreating log10 for {len(selected_period_cols_sorted)} spectral periods...")
for col in tqdm(selected_period_cols_sorted, desc="Log-transforming Sa"):
    df[f'log10_{col}'] = np.log10(df[col])

# Verify no inf/nan values after log transformation
print("\nVerifying data integrity...")
log_feature_cols = ['log10_Rjb_km', 'log10_Vs30']
log_target_cols = ['log10_PGA_g', 'log10_PGV_cm_sec'] + [f'log10_{col}' for col in selected_period_cols_sorted]

# Check for inf/nan in log-transformed columns
inf_nan_count = 0
print("Checking for inf/nan values in log-transformed columns...")
for col in tqdm(log_feature_cols + log_target_cols, desc="Validating columns"):
    invalid_mask = ~np.isfinite(df[col])
    if invalid_mask.any():
        inf_nan_count += invalid_mask.sum()
        df = df[~invalid_mask]

if inf_nan_count > 0:
    print(f"  ⚠ Removed {inf_nan_count} rows with inf/nan values")
    df = df.reset_index(drop=True)
else:
    print(f"  ✓ All values are finite and valid")

# Store column lists for later use
input_feature_cols = ['Earthquake_Magnitude', 'Rjb_km', 'log10_Rjb_km', 'log10_Vs30', 'Fault_Type']
output_target_cols = log_target_cols


DATA PREPROCESSING

Creating log-transformed input features...
  ✓ log10_Rjb_km, ✓ log10_Vs30
  ✓ log10_PGA_g, ✓ log10_PGV_cm_sec

Creating log10 for 90 spectral periods...


Log-transforming Sa: 100%|██████████| 90/90 [00:00<00:00, 325.30it/s]



Verifying data integrity...
Checking for inf/nan values in log-transformed columns...


Validating columns: 100%|██████████| 94/94 [00:00<00:00, 4163.83it/s]

  ✓ All values are finite and valid


In [7]:
# Train/test split and feature scaling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("="*80)
print("TRAIN-TEST SPLIT")
print("="*80)

# Prepare X and y from dataframe
X = df[input_feature_cols].values
y = df[output_target_cols].values

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target matrix shape: {y.shape}")

# Split data (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\n✓ Training samples: {X_train.shape[0]:,}")
print(f"✓ Testing samples: {X_test.shape[0]:,}")
print(f"✓ Split ratio: 80% train / 20% test")

# Standardize features (critical for neural networks)
print("\nApplying feature standardization...")
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled  = scaler_X.transform(X_test)

# Targets are already in log10 space (output_target_cols = log10_PGA_g, log10_Sa...).
# Just apply StandardScaler to normalise the log10 values — no extra log needed.
scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled  = scaler_y.transform(y_test)

print(f"✓ X_train_scaled: {X_train_scaled.shape}")
print(f"✓ y_train_scaled: {y_train_scaled.shape}  (log10 → StandardScaler)")
print(f"✓ Scalers fitted and ready")

print("\n" + "="*80)
print("DATA PREPARATION COMPLETE")
print("="*80)


TRAIN-TEST SPLIT

Feature matrix shape: (54967, 5)
Target matrix shape: (54967, 92)

✓ Training samples: 43,973
✓ Testing samples: 10,994
✓ Split ratio: 80% train / 20% test

Applying feature standardization...
✓ X_train_scaled: (43973, 5)
✓ y_train_scaled: (43973, 92)  (log10 → StandardScaler)
✓ Scalers fitted and ready

DATA PREPARATION COMPLETE


In [8]:
class LSTM(nn.Module):
    """
    Hybrid LSTM model for GMPE (improved).

    Architecture:
    - PGA/PGV : Linear(input → hidden) → ReLU → Dropout → Linear(hidden → 1)
    - Sa      : 1-layer LSTM  (input = [prev_Sa | base_features])
                → ReLU head  Linear(hidden+input → hidden) → ReLU → Linear(hidden → 1)

    Key improvements over v1:
      1. ReLU hidden layer in PGA/PGV heads  → captures non-linear attenuation
      2. Base features fed into LSTM at every step  → hidden state conditioned on
         earthquake context throughout the sequence, not just at the output
      3. ReLU hidden layer in Sa predictor  → same non-linearity benefit
    """
    def __init__(self, input_size=5, hidden_size=64, num_sa_outputs=90, dropout_rate=0.2):
        super(LSTM, self).__init__()

        self.input_size     = input_size
        self.hidden_size    = hidden_size
        self.num_sa_outputs = num_sa_outputs
        self.dropout_rate   = dropout_rate

        # PGA / PGV heads: Linear → ReLU → Dropout → Linear
        self.pga_head = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size, 1)
        )
        self.pgv_head = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size, 1)
        )

        # 1-layer LSTM: input = [prev_Sa (1) + base_features (input_size)]
        # Feeding features at every step lets the hidden state evolve with
        # full earthquake context, not just the Sa sequence shape.
        self.lstm = nn.LSTM(
            input_size=1 + input_size,   # prev Sa + base features
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )

        # Sa predictor: Linear(hidden+input → hidden) → ReLU → Dropout → Linear(hidden → 1)
        self.sa_predictor = nn.Sequential(
            nn.Linear(hidden_size + input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x, y_sa=None):
        """
        Args:
            x   : (batch, input_size) — base features
            y_sa: (batch, num_sa_outputs) — ground-truth Sa for teacher-forcing
                  If None → autoregressive inference loop
        Returns:
            (batch, 2 + num_sa_outputs) — [PGA, PGV, Sa_1 … Sa_N]
        """
        batch_size = x.size(0)
        dev        = x.device

        # Static heads
        pga = self.pga_head(x)   # (batch, 1)
        pgv = self.pgv_head(x)   # (batch, 1)

        if y_sa is not None:
            # ── TRAINING: teacher-forcing (single LSTM call) ──────────────
            # Shift right: [0, Sa1, …, Sa_{N-1}] → predicts [Sa1, …, Sa_N]
            prev_sa_seq = torch.cat([
                torch.zeros(batch_size, 1, device=dev),
                y_sa[:, :-1]
            ], dim=1).unsqueeze(2)                             # (batch, N, 1)

            # Concat base features at every LSTM step
            x_expanded = x.unsqueeze(1).expand(-1, self.num_sa_outputs, -1)  # (batch, N, input_size)
            lstm_input = torch.cat([prev_sa_seq, x_expanded], dim=2)          # (batch, N, 1+input_size)

            lstm_out, _ = self.lstm(lstm_input)                # (batch, N, hidden)

            combined  = torch.cat([lstm_out, x_expanded], dim=2)  # (batch, N, hidden+input)
            sa_tensor = self.sa_predictor(combined).squeeze(-1)   # (batch, N)
        else:
            # ── INFERENCE: autoregressive loop ───────────────────────────
            sa_predictions = []
            h       = torch.zeros(1, batch_size, self.hidden_size, device=dev)
            c       = torch.zeros(1, batch_size, self.hidden_size, device=dev)
            prev_sa = torch.zeros(batch_size, device=dev)      # (batch,)

            for _ in range(self.num_sa_outputs):
                # LSTM input: [prev_sa, x] at each step  → (batch, 1, 1+input_size)
                lstm_input       = torch.cat([prev_sa.unsqueeze(1).unsqueeze(2),
                                              x.unsqueeze(1)], dim=2)
                lstm_out, (h, c) = self.lstm(lstm_input, (h, c))

                combined = torch.cat([lstm_out.squeeze(1), x], dim=1)  # (batch, hidden+input)
                sa_i     = self.sa_predictor(combined).squeeze(-1)      # (batch,)
                sa_predictions.append(sa_i)
                prev_sa = sa_i

            sa_tensor = torch.stack(sa_predictions, dim=1)  # (batch, N)

        return torch.cat([pga, pgv, sa_tensor], dim=1)      # (batch, 2+N)


print("✓ LSTM model defined  (improved v2)")
print("  PGA/PGV  : Linear(input→H) → ReLU → Dropout → Linear(H→1)")
print("  LSTM     : 1 layer, input=[prev_Sa | base_features]")
print("  Sa head  : Linear(H+input→H) → ReLU → Dropout → Linear(H→1)")
print("  Training : teacher-forcing  |  Inference: autoregressive loop")


✓ LSTM model defined  (improved v2)
  PGA/PGV  : Linear(input→H) → ReLU → Dropout → Linear(H→1)
  LSTM     : 1 layer, input=[prev_Sa | base_features]
  Sa head  : Linear(H+input→H) → ReLU → Dropout → Linear(H→1)
  Training : teacher-forcing  |  Inference: autoregressive loop


In [9]:
def train(X_train, y_train, X_val, y_val, 
          hidden_size=20, lr=0.001, batch_size=64, 
          num_epochs=50, patience=10, dropout_rate=0.2, verbose=False):
    """
    Train LSTM model with given hyperparameters.
    
    Args:
        X_train, y_train: Training data (scaled)
        X_val, y_val: Validation data (scaled)
        hidden_size: LSTM hidden dimension
        lr: Learning rate
        batch_size: Mini-batch size
        epochs: Maximum training epochs
        patience: Early stopping patience
        verbose: Print training progress
        
    Returns:
        dict: {
            'model': trained model,
            'train_loss': final training loss,
            'val_loss': final validation loss,
            'best_epoch': epoch with best val loss,
            'train_history': loss per epoch,
            'val_history': val loss per epoch
        }
    """
    # Convert to tensors
    X_train_tensor = torch.FloatTensor(X_train).to(device)
    y_train_tensor = torch.FloatTensor(y_train).to(device)
    X_val_tensor = torch.FloatTensor(X_val).to(device)
    y_val_tensor = torch.FloatTensor(y_val).to(device)
    
    # Create DataLoader
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    # Initialize model
    model = LSTM(input_size=X_train.shape[1], hidden_size=hidden_size, num_sa_outputs=y_train.shape[1]-2, dropout_rate=dropout_rate).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    
    # Training tracking
    train_history = []
    val_history = []
    # Initialise to model's untrained weights so load_state_dict never receives None
    # (guards against NaN val_loss on epoch 1 where NaN < inf is False)
    best_model_state = copy.deepcopy(model.state_dict())
    best_epoch = 0
    patience_counter = 0
    best_val_loss = float('inf')
    
    # Training loop
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            # Pass ground truth Sa (columns 2:) as LSTM input
            outputs = model(batch_X, y_sa=batch_y[:, 2:])
            loss = criterion(outputs, batch_y)
            loss.backward()
            # Clip gradients — prevents NaN losses from LSTM exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        train_history.append(train_loss)
        
        # Validation phase
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val_tensor, y_sa=y_val_tensor[:, 2:])
            val_loss = criterion(val_outputs, y_val_tensor).item()
        val_history.append(val_loss)
        
        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
        
        # Print progress
        if verbose and (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")
        
        # Early stopping
        if patience_counter >= patience:
            if verbose:
                print(f"Early stopping at epoch {epoch+1}")
            break
    
    # Load best model
    model.load_state_dict(best_model_state)
    
    return {
        'model': model,
        'train_loss': train_history[best_epoch],
        'val_loss': best_val_loss,
        'best_epoch': best_epoch,
        'train_history': train_history,
        'val_history': val_history
    }


print("✓ Training function defined")
print("✓ Training function defined")

✓ Training function defined
✓ Training function defined


In [10]:
# Install and import wandb
import wandb

# Split train into train/val for hyperparameter search
X_train_part, X_val, y_train_part, y_val = train_test_split(
    X_train_scaled, y_train_scaled, test_size=0.2, random_state=42
)

print(f"✓ Train set: {X_train_part.shape[0]:,} samples")
print(f"✓ Validation set: {X_val.shape[0]:,} samples")
print(f"✓ Test set: {X_test_scaled.shape[0]:,} samples (held out for final evaluation)")


✓ Train set: 35,178 samples
✓ Validation set: 8,795 samples
✓ Test set: 10,994 samples (held out for final evaluation)


# Hyperparameter Search (Sequential)

We'll search for optimal hyperparameters sequentially:
1. **Step 1:** Find best batch_size (fix lr=0.002, hidden_size=10)
2. **Step 2:** Find best learning_rate (use best batch_size, fix hidden_size=10)
3. **Step 3:** Find best hidden_size (use best batch_size and learning_rate)

In [11]:
# Initialize W&B for sequential search
wandb.init(project="tutorial4-lstm", name="sequential_hyperparam_search", reinit=True)

# Define search space
batch_sizes    = [16, 32, 64, 128]
learning_rates = [0.002, 0.001, 0.0005, 0.0001]
hidden_sizes   = [10, 15, 20, 25, 30]

# Default hidden size used while searching batch size / learning rate
default_hs = 20

# Training hyperparameters
num_epochs = 100
patience   = 15

print("="*80)
print("SEQUENTIAL HYPERPARAMETER SEARCH")
print("="*80)
print(f"Search space:")
print(f"  Batch sizes:    {batch_sizes}")
print(f"  Learning rates: {learning_rates}")
print(f"  Hidden sizes:   {hidden_sizes}")
print(f"\nDefault fixed hidden size for steps 1 & 2: {default_hs}")
print(f"\nTraining config:")
print(f"  Max epochs: {num_epochs}")
print(f"  Early stopping patience: {patience}")
print(f"  Total runs: {len(batch_sizes) + len(learning_rates) + len(hidden_sizes)}")
print("="*80)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/prem/.netrc.
wandb: Currently logged in as: premsuggu11 (prem11suggu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


SEQUENTIAL HYPERPARAMETER SEARCH
Search space:
  Batch sizes:    [16, 32, 64, 128]
  Learning rates: [0.002, 0.001, 0.0005, 0.0001]
  Hidden sizes:   [10, 15, 20, 25, 30]

Default fixed hidden size for steps 1 & 2: 20

Training config:
  Max epochs: 100
  Early stopping patience: 15
  Total runs: 13


In [12]:
# STEP 1: Search for best batch size
print("\n" + "="*80)
print("STEP 1: SEARCHING BATCH SIZE")
print("="*80)
print(f"Fixed: lr={learning_rates[0]}, hidden_size={default_hs}")

bs_results = []
for bs in tqdm(batch_sizes, desc="Batch size search"):
    result = train(
        X_train_part, y_train_part, X_val, y_val,
        hidden_size=default_hs,
        lr=learning_rates[0],
        batch_size=bs,
        num_epochs=num_epochs,
        patience=patience,
        verbose=False
    )

    bs_results.append({
        'batch_size': bs,
        'val_loss':   result['val_loss'],
        'train_loss': result['train_loss'],
        'best_epoch': result['best_epoch']
    })

    print(f"  batch_size={bs:3d} → val_loss={result['val_loss']:.6f}, epochs={result['best_epoch']}")

# Find best batch size
best_bs      = min(bs_results, key=lambda x: x['val_loss'])['batch_size']
best_bs_loss = min(bs_results, key=lambda x: x['val_loss'])['val_loss']

# Log to W&B
bs_table = wandb.Table(
    data=[[r['batch_size'], r['train_loss'], r['val_loss'], r['best_epoch']] for r in bs_results],
    columns=["batch_size", "train_loss", "val_loss", "best_epoch"]
)
wandb.log({"batch_size_search": wandb.plot.line(bs_table, "batch_size", "val_loss",
           title="Batch Size vs Validation Loss")})
wandb.run.summary.update({"best_batch_size": best_bs, "best_bs_val_loss": best_bs_loss})

print(f"\n✓ Best batch_size: {best_bs} (val_loss={best_bs_loss:.6f})")



STEP 1: SEARCHING BATCH SIZE
Fixed: lr=0.002, hidden_size=20


Batch size search:  25%|██▌       | 1/4 [03:56<11:50, 236.77s/it]

  batch_size= 16 → val_loss=0.013205, epochs=7


Batch size search:  50%|█████     | 2/4 [06:02<05:42, 171.20s/it]

  batch_size= 32 → val_loss=0.013252, epochs=9


Batch size search:  75%|███████▌  | 3/4 [07:32<02:14, 134.31s/it]

  batch_size= 64 → val_loss=0.012863, epochs=19


Batch size search: 100%|██████████| 4/4 [08:26<00:00, 126.59s/it]

  batch_size=128 → val_loss=0.012933, epochs=20



✓ Best batch_size: 64 (val_loss=0.012863)


In [ ]:
# STEP 2: Search for best learning rate
print("\n" + "="*80)
print("STEP 2: SEARCHING LEARNING RATE")
print("="*80)
print(f"Fixed: batch_size={best_bs}, hidden_size={default_hs}")

lr_results = []
for lr in tqdm(learning_rates, desc="Learning rate search"):
    result = train(
        X_train_part, y_train_part, X_val, y_val,
        hidden_size=default_hs,
        lr=lr,
        batch_size=best_bs,
        num_epochs=num_epochs,
        patience=patience,
        verbose=False
    )

    lr_results.append({
        'learning_rate': lr,
        'val_loss':      result['val_loss'],
        'train_loss':    result['train_loss'],
        'best_epoch':    result['best_epoch']
    })

    print(f"  lr={lr:.6f} → val_loss={result['val_loss']:.6f}, epochs={result['best_epoch']}")

# Find best learning rate
best_lr      = min(lr_results, key=lambda x: x['val_loss'])['learning_rate']
best_lr_loss = min(lr_results, key=lambda x: x['val_loss'])['val_loss']

# Log to W&B
lr_table = wandb.Table(
    data=[[r['learning_rate'], r['train_loss'], r['val_loss'], r['best_epoch']] for r in lr_results],
    columns=["learning_rate", "train_loss", "val_loss", "best_epoch"]
)
wandb.log({"learning_rate_search": wandb.plot.line(lr_table, "learning_rate", "val_loss",
           title="Learning Rate vs Validation Loss")})
wandb.run.summary.update({"best_learning_rate": best_lr, "best_lr_val_loss": best_lr_loss})

print(f"\n✓ Best learning_rate: {best_lr} (val_loss={best_lr_loss:.6f})")



STEP 2: SEARCHING LEARNING RATE
Fixed: batch_size=64, hidden_size=20


Learning rate search:  25%|██▌       | 1/4 [01:19<03:58, 79.47s/it]

  lr=0.002000 → val_loss=0.013095, epochs=13


In [ ]:
# STEP 3: Search for best hidden size
print("\n" + "="*80)
print("STEP 3: SEARCHING HIDDEN SIZE")
print("="*80)
print(f"Fixed: batch_size={best_bs}, lr={best_lr}")

hs_results = []
for hs in tqdm(hidden_sizes, desc="Hidden size search"):
    result = train(
        X_train_part, y_train_part, X_val, y_val,
        hidden_size=hs,
        lr=best_lr,
        batch_size=best_bs,
        num_epochs=num_epochs,
        patience=patience,
        verbose=False
    )
    
    hs_results.append({
        'hidden_size': hs,
        'val_loss': result['val_loss'],
        'train_loss': result['train_loss'],
        'best_epoch': result['best_epoch']
    })
    
    print(f"  hidden_size={hs:2d} → val_loss={result['val_loss']:.6f}, epochs={result['best_epoch']}")

# Find best hidden size
best_hs = min(hs_results, key=lambda x: x['val_loss'])['hidden_size']
best_hs_loss = min(hs_results, key=lambda x: x['val_loss'])['val_loss']

# Log to W&B as a Table
hs_table = wandb.Table(
    data=[[r['hidden_size'], r['train_loss'], r['val_loss'], r['best_epoch']] for r in hs_results],
    columns=["hidden_size", "train_loss", "val_loss", "best_epoch"]
)
wandb.log({"hidden_size_search": wandb.plot.line(hs_table, "hidden_size", "val_loss",
           title="Hidden Size vs Validation Loss")})
wandb.run.summary.update({"best_hidden_size": best_hs, "best_hs_val_loss": best_hs_loss})

print(f"\n✓ Best hidden_size: {best_hs} (val_loss={best_hs_loss:.6f})")

In [ ]:
# Display final best configuration
print("\n" + "="*80)
print("BEST CONFIGURATION")
print("="*80)
print(f"Batch size:     {best_bs}")
print(f"Learning rate:  {best_lr}")
print(f"Hidden size:    {best_hs}")
print(f"Validation loss: {best_hs_loss:.6f}")
print("="*80)

# Log final config to W&B
wandb.summary['final_batch_size'] = best_bs
wandb.summary['final_learning_rate'] = best_lr
wandb.summary['final_hidden_size'] = best_hs
wandb.summary['final_val_loss'] = best_hs_loss

# Finish W&B run
wandb.finish()

print("\n✓ Hyperparameter search complete!")
print(f"✓ View results at: https://wandb.ai/prem11suggu/tutorial4-lstm")

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Plot 1: Batch Size vs Loss ──────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

ax.plot([r['batch_size'] for r in bs_results],
        [r['train_loss'] for r in bs_results],
        marker='o', linestyle='-', linewidth=2.5, markersize=10,
        label='Training Loss', color='#3498DB')
ax.plot([r['batch_size'] for r in bs_results],
        [r['val_loss'] for r in bs_results],
        marker='s', linestyle='--', linewidth=2.5, markersize=10,
        label='Validation Loss', color='#E74C3C')
ax.scatter([best_bs], [best_bs_loss], s=400, c='gold', marker='*',
           edgecolors='black', linewidth=2, zorder=5,
           label=f'Best: {best_bs}')

ax.set_xlabel('Batch Size', fontsize=14, fontweight='bold')
ax.set_ylabel('MSE Loss (scaled)', fontsize=14, fontweight='bold')
ax.set_title('Batch Size Effect on Model Performance\n'
             f'(LSTM, hidden={default_hs}, lr={learning_rates[0]})',
             fontsize=16, fontweight='bold')
ax.set_xticks([r['batch_size'] for r in bs_results])
ax.legend(fontsize=12, loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/batch_size_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {OUTPUT_DIR}/batch_size_analysis.png")

# ── Plot 2: Learning Rate vs Loss ───────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

ax.plot([r['learning_rate'] for r in lr_results],
        [r['train_loss'] for r in lr_results],
        marker='o', linestyle='-', linewidth=2.5, markersize=10,
        label='Training Loss', color='#3498DB')
ax.plot([r['learning_rate'] for r in lr_results],
        [r['val_loss'] for r in lr_results],
        marker='s', linestyle='--', linewidth=2.5, markersize=10,
        label='Validation Loss', color='#E74C3C')
ax.scatter([best_lr], [best_lr_loss], s=400, c='gold', marker='*',
           edgecolors='black', linewidth=2, zorder=5,
           label=f'Best: {best_lr}')

ax.set_xlabel('Learning Rate', fontsize=14, fontweight='bold')
ax.set_ylabel('MSE Loss (scaled)', fontsize=14, fontweight='bold')
ax.set_title('Learning Rate Effect on Model Performance\n'
             f'(LSTM, batch={best_bs}, hidden={default_hs})',
             fontsize=16, fontweight='bold')
ax.set_xscale('log')
ax.legend(fontsize=12, loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/learning_rate_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {OUTPUT_DIR}/learning_rate_analysis.png")

# ── Plot 3: Hidden Size vs Loss ─────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

ax.plot([r['hidden_size'] for r in hs_results],
        [r['train_loss'] for r in hs_results],
        marker='o', linestyle='-', linewidth=2.5, markersize=10,
        label='Training Loss', color='#3498DB')
ax.plot([r['hidden_size'] for r in hs_results],
        [r['val_loss'] for r in hs_results],
        marker='s', linestyle='--', linewidth=2.5, markersize=10,
        label='Validation Loss', color='#E74C3C')
ax.scatter([best_hs], [best_hs_loss], s=400, c='gold', marker='*',
           edgecolors='black', linewidth=2, zorder=5,
           label=f'Best: {best_hs}')

ax.set_xlabel('Hidden Size', fontsize=14, fontweight='bold')
ax.set_ylabel('MSE Loss (scaled)', fontsize=14, fontweight='bold')
ax.set_title('Hidden Size Effect on Model Performance\n'
             f'(LSTM, batch={best_bs}, lr={best_lr})',
             fontsize=16, fontweight='bold')
ax.set_xticks([r['hidden_size'] for r in hs_results])
ax.legend(fontsize=12, loc='best', framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/hidden_size_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {OUTPUT_DIR}/hidden_size_analysis.png")


In [ ]:
# Dropout Comparison: Best config with vs without dropout
print("="*80)
print("DROPOUT COMPARISON")
print("="*80)
print(f"Best config: batch={best_bs}, lr={best_lr}, hidden={best_hs}")
print(f"Comparing: dropout=0.2  vs  dropout=0.0\n")

# Train WITH dropout (0.2)
print("Training WITH dropout (0.2)...")
result_with_dropout = train(
    X_train_part, y_train_part, X_val, y_val,
    hidden_size=best_hs,
    lr=best_lr,
    batch_size=best_bs,
    num_epochs=num_epochs,
    patience=patience,
    dropout_rate=0.2,
    verbose=True
)

# Train WITHOUT dropout (0.0)
print("\nTraining WITHOUT dropout (0.0)...")
result_no_dropout = train(
    X_train_part, y_train_part, X_val, y_val,
    hidden_size=best_hs,
    lr=best_lr,
    batch_size=best_bs,
    num_epochs=num_epochs,
    patience=patience,
    dropout_rate=0.0,
    verbose=True
)

print(f"\n{'='*80}")
print(f"DROPOUT COMPARISON RESULTS")
print(f"{'='*80}")
print(f"With dropout (0.2):    train={result_with_dropout['train_loss']:.6f}, val={result_with_dropout['val_loss']:.6f}")
print(f"Without dropout (0.0): train={result_no_dropout['train_loss']:.6f}, val={result_no_dropout['val_loss']:.6f}")

# --- Plot: Validation loss curves over epochs ---
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

epochs_d  = range(1, len(result_with_dropout['val_history'])  + 1)
epochs_nd = range(1, len(result_no_dropout['val_history']) + 1)

ax.plot(epochs_d,  result_with_dropout['train_history'],
        linestyle='-',  linewidth=2.5, color='#E74C3C', label='Train Loss (dropout=0.2)')
ax.plot(epochs_d,  result_with_dropout['val_history'],
        linestyle='--', linewidth=2.5, color='#E74C3C', label='Val Loss   (dropout=0.2)')
ax.plot(epochs_nd, result_no_dropout['train_history'],
        linestyle='-',  linewidth=2.5, color='#3498DB', label='Train Loss (no dropout)')
ax.plot(epochs_nd, result_no_dropout['val_history'],
        linestyle='--', linewidth=2.5, color='#3498DB', label='Val Loss   (no dropout)')

ax.axvline(result_with_dropout['best_epoch'] + 1,  color='#E74C3C', linestyle=':', alpha=0.6,
           label=f'Best epoch (dropout): {result_with_dropout["best_epoch"]+1}')
ax.axvline(result_no_dropout['best_epoch'] + 1, color='#3498DB', linestyle=':', alpha=0.6,
           label=f'Best epoch (no dropout): {result_no_dropout["best_epoch"]+1}')

ax.set_xlabel('Epoch', fontsize=14, fontweight='bold')
ax.set_ylabel('MSE Loss (scaled)', fontsize=14, fontweight='bold')
ax.set_title(f'Effect of Dropout on Training Dynamics\n'
             f'(Best Config: batch={best_bs}, lr={best_lr}, hidden={best_hs})',
             fontsize=16, fontweight='bold')
ax.legend(fontsize=11, loc='upper right', framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/dropout_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {OUTPUT_DIR}/dropout_comparison.png")


# Model Evaluation & Residual Analysis

Use the best model from the dropout comparison to evaluate on the held-out test set and compute binned residuals.


In [ ]:
# Pick the better model from the dropout comparison (already trained)
print("="*80)
print("MODEL EVALUATION ON TEST SET")
print("="*80)

if result_with_dropout['val_loss'] <= result_no_dropout['val_loss']:
    best_dropout = 0.2
    final_model  = result_with_dropout['model']
    print(f"✓ Using dropout=0.2 model (val_loss={result_with_dropout['val_loss']:.6f})")
else:
    best_dropout = 0.0
    final_model  = result_no_dropout['model']
    print(f"✓ Using dropout=0.0 model (val_loss={result_no_dropout['val_loss']:.6f})")

print(f"Config: batch={best_bs}, lr={best_lr}, hidden={best_hs}, dropout={best_dropout}")

# Generate predictions on test set using INFERENCE mode (y_sa=None → autoregressive)
# This gives honest metrics — the model predicts Sa step-by-step without seeing ground truth.
final_model.eval()
X_test_tensor = torch.FloatTensor(X_test_scaled).to(device)
y_test_tensor = torch.FloatTensor(y_test_scaled).to(device)

with torch.no_grad():
    y_pred_scaled = final_model(X_test_tensor, y_sa=None).cpu().numpy()  # true inference mode

# Compute residuals (in scaled space)
residuals_test = y_pred_scaled - y_test_scaled   # (n_test, 92)

# Inverse-transform for original-scale metrics
# Inverse-transform: undo StandardScaler then undo log10  (10 **)
y_pred_original = 10 ** scaler_y.inverse_transform(y_pred_scaled)
y_test_original = 10 ** scaler_y.inverse_transform(y_test_scaled)

# Overall metrics
r2_overall  = r2_score(y_test_scaled, y_pred_scaled)
mae_overall = mean_absolute_error(y_test_scaled, y_pred_scaled)
print(f"\n{'='*80}")
print(f"TEST SET METRICS  (autoregressive inference — no teacher forcing)")
print(f"{'='*80}")
print(f"  Overall R²:  {r2_overall:.4f}")
print(f"  Overall MAE: {mae_overall:.6f} (scaled)")
print(f"  Predictions shape: {y_pred_scaled.shape}")
print(f"  Residuals shape:   {residuals_test.shape}")
print(f"{'='*80}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# BINNED RESIDUAL PLOTS — PGA residuals vs. each predictor
# ═══════════════════════════════════════════════════════════════════════════════

# PGA residuals (first output column)
residuals_pga = residuals_test[:, 0]

# Unscale test features for x-axis labels
X_test_unscaled = scaler_X.inverse_transform(X_test_scaled)
magnitude_test  = X_test_unscaled[:, 0]
rjb_test        = X_test_unscaled[:, 1]

# Recover raw Vs30 from log10_Vs30 (column index 3)
vs30_test = 10 ** X_test_unscaled[:, 3]

# ─── Helper: create binned statistics ────────────────────────────────────────
def bin_residuals(feature, residuals, n_bins=15, log_scale=False):
    """Bin residuals by feature value, return centres, means, stds, counts."""
    mask = np.isfinite(feature) & np.isfinite(residuals)
    if log_scale:
        mask &= (feature > 0)
    feat, res = feature[mask], residuals[mask]

    if log_scale:
        edges = np.logspace(np.log10(feat.min()), np.log10(feat.max()), n_bins + 1)
    else:
        edges = np.linspace(feat.min(), feat.max(), n_bins + 1)

    idx = np.digitize(feat, edges)
    centres, means, stds, counts = [], [], [], []
    for b in range(1, len(edges)):
        m = idx == b
        if m.sum() >= 5:
            if log_scale:
                centres.append(10 ** ((np.log10(edges[b-1]) + np.log10(edges[b])) / 2))
            else:
                centres.append((edges[b-1] + edges[b]) / 2)
            means.append(res[m].mean())
            stds.append(res[m].std())
            counts.append(m.sum())
    return np.array(centres), np.array(means), np.array(stds), np.array(counts)

# ─── Helper: draw one polished binned-residual panel ─────────────────────────
def plot_binned_residual(ax, feature, residuals, centres, means, stds, counts,
                         xlabel, color_scatter, color_bin, log_x=False):
    """Draw a clean binned residual plot on the given axes."""
    ax.scatter(feature, residuals, s=12, alpha=0.35,
               color=color_scatter, edgecolors='none', rasterized=True, zorder=1)
    ax.errorbar(centres, means, yerr=stds,
                fmt='D', markersize=7, capsize=4, capthick=1.8, linewidth=0,
                elinewidth=1.8, color=color_bin, markeredgecolor='white',
                markeredgewidth=0.8, label='Bin mean ± 1σ', zorder=4)
    ax.plot(centres, means, '-', linewidth=1.2, color=color_bin, alpha=0.6, zorder=3)
    ax.axhline(0, color='#2C3E50', linewidth=1.4, linestyle='--', alpha=0.7, zorder=2)
    global_std = residuals[np.isfinite(residuals)].std()
    ax.axhspan(-global_std, global_std, color='#BDC3C7', alpha=0.12, zorder=0)

    if log_x:
        ax.set_xscale('log')
        valid_feat = feature[(feature > 0) & np.isfinite(feature)]
        x_lo = 10 ** (np.log10(valid_feat.min()) - 0.08)
        x_hi = 10 ** (np.log10(valid_feat.max()) + 0.08)
        ax.set_xlim([x_lo, x_hi])
        ax.grid(True, which='major', alpha=0.25, linewidth=0.6)
        ax.grid(True, which='minor', alpha=0.10, linewidth=0.4)
    else:
        span = feature.max() - feature.min()
        ax.set_xlim([feature.min() - 0.03 * span, feature.max() + 0.03 * span])
        ax.grid(True, alpha=0.25, linewidth=0.6)

    ax.set_xlabel(xlabel, fontsize=13, fontweight='bold')
    ax.set_ylabel('Residual  (predicted − actual)', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11, framealpha=0.9, edgecolor='#BDC3C7')
    ax.tick_params(labelsize=11)

# ─── Create the three plots ──────────────────────────────────────────────────
plot_configs = [
    ('Earthquake Magnitude',               magnitude_test, False, '#81C784', '#D32F2F', 'residual_vs_magnitude.png'),
    ('Joyner-Boore Distance, $R_{JB}$ (km)', rjb_test,    True,  '#90CAF9', '#1565C0', 'residual_vs_distance.png'),
    ('$V_{S30}$ (m/s)',                    vs30_test,      True,  '#CE93D8', '#6A1B9A', 'residual_vs_vs30.png'),
]

c_mag,  m_mag,  s_mag,  n_mag  = bin_residuals(magnitude_test, residuals_pga, 15, False)
c_rjb,  m_rjb,  s_rjb,  n_rjb  = bin_residuals(rjb_test,       residuals_pga, 15, True)
c_vs30, m_vs30, s_vs30, n_vs30  = bin_residuals(vs30_test,      residuals_pga, 15, True)

binned_data = [
    (c_mag,  m_mag,  s_mag,  n_mag),
    (c_rjb,  m_rjb,  s_rjb,  n_rjb),
    (c_vs30, m_vs30, s_vs30, n_vs30),
]

for (xlabel, feat, log_x, c_scat, c_bin, fname), (centres, means, stds, counts) in zip(plot_configs, binned_data):
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.patch.set_facecolor('white')
    plot_binned_residual(ax, feat, residuals_pga, centres, means, stds, counts,
                         xlabel, c_scat, c_bin, log_x)
    var_label = xlabel.split('(')[0].strip().replace('$', '').replace('{', '').replace('}', '')
    ax.set_title(f'Binned Residuals: PGA vs {var_label}\n'
                 f'({len(centres)} bins, {int(counts.sum()):,} points)',
                 fontsize=15, fontweight='bold', pad=12)
    plt.tight_layout()
    save_path = f'{OUTPUT_DIR}/{fname}'
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"✓ Saved: {save_path}")

# ─── Summary table ───────────────────────────────────────────────────────────
print(f"\n{'='*80}")
print("BINNED RESIDUAL SUMMARY")
print(f"{'='*80}")
for name, centres, means, stds, counts in [
    ('Magnitude',       c_mag,  m_mag,  s_mag,  n_mag),
    ('Distance (Rjb)',  c_rjb,  m_rjb,  s_rjb,  n_rjb),
    ('Vs30',            c_vs30, m_vs30, s_vs30, n_vs30),
]:
    print(f"\n  {name}:")
    print(f"    Bins: {len(centres)}  |  Points: {int(counts.sum()):,}")
    print(f"    Mean residual range: [{means.min():.4f}, {means.max():.4f}]")
    print(f"    Std range:           [{stds.min():.4f}, {stds.max():.4f}]")
print(f"\n{'='*80}")


# Parametric Study & Feature Importance

In [ ]:

# ── Periods array from loaded data ─────────────────────────────────────────
periods = [float(col[1:-1].replace('pt', '.')) for col in selected_period_cols_sorted]

# ── Shared prediction helper ───────────────────────────────────────────────
def predict_spectrum(mag, rjb, vs30, fault_type=0):
    """
    Predict Sa spectrum (g) via autoregressive LSTM inference.
    Calls model with y_sa=None → full autoregressive loop (no teacher forcing).
    """
    final_model.eval()   # ensure dropout/BN are in inference mode
    X_raw = np.array([[mag, rjb, np.log10(rjb), np.log10(vs30), fault_type]])
    with torch.no_grad():
        y_s = final_model(
            torch.FloatTensor(scaler_X.transform(X_raw)).to(device),
            y_sa=None        # ← autoregressive: previous Sa fed back at each step
        ).cpu().numpy()
    return 10 ** scaler_y.inverse_transform(y_s)[0, 2:]   # Sa spectrum in g  (undo log10)

# ── Shared plot helper ─────────────────────────────────────────────────────
def plot_spectra(sa_list, labels, colors, marker, title_var, fixed_desc, filename):
    """Plot response spectra as log-log and semi-log subplots."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('white')
    for sa, lbl, clr in zip(sa_list, labels, colors):
        kw = dict(marker=marker, linestyle='-', linewidth=2.5, markersize=5, label=lbl, color=clr)
        ax1.loglog(periods, sa, **kw)
        ax2.semilogy(periods, sa, **kw)
    for ax in (ax1, ax2):
        ax.set_xlabel('Period (s)', fontsize=13, fontweight='bold')
        ax.set_ylabel('Sa (g)', fontsize=13, fontweight='bold')
        ax.set_xlim(0.01, 4.1)
        ax.grid(True, alpha=0.3, which='both')
        ax.legend(fontsize=10, framealpha=0.9)
        ax.tick_params(labelsize=11)
    ax1.set_title('Response Spectra (Log-Log)', fontsize=14, fontweight='bold')
    ax2.set_title(f'{title_var} Variation\n({fixed_desc})', fontsize=14, fontweight='bold')
    plt.tight_layout()
    path = f'{OUTPUT_DIR}/{filename}'
    plt.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"✓ Saved: {path}")

# ── 1. Magnitude sweep ─────────────────────────────────────────────────────
mags   = [4.0, 5.0, 6.0, 7.0, 7.5]
c_mags = ['#E74C3C', '#27AE60', '#3498DB', '#9B59B6', '#E91E63']
plot_spectra(
    [predict_spectrum(m, 10.0, 760.0) for m in mags],
    [f'Mw {m}' for m in mags], c_mags, 'o',
    'Magnitude', 'Rjb = 10 km, Vs30 = 760 m/s',
    'parametric_magnitude.png'
)

# ── 2. Distance sweep ──────────────────────────────────────────────────────
dists   = [5.0, 10.0, 30.0, 80.0, 200.0]
c_dists = ['#E74C3C', '#F39C12', '#F1C40F', '#27AE60', '#3498DB']
plot_spectra(
    [predict_spectrum(6.5, d, 760.0) for d in dists],
    [f'Rjb = {d:.0f} km' for d in dists], c_dists, 's',
    'Distance (Rjb)', 'Mw = 6.5, Vs30 = 760 m/s',
    'parametric_distance.png'
)

# ── 3. Vs30 sweep ──────────────────────────────────────────────────────────
vs30s     = [150, 270, 450, 760, 1500]
vs30_lbls = ['E (150 m/s)', 'D (270 m/s)', 'C (450 m/s)', 'B (760 m/s)', 'A (1500 m/s)']
c_vs30s   = ['#E74C3C', '#F39C12', '#F1C40F', '#27AE60', '#3498DB']
plot_spectra(
    [predict_spectrum(6.5, 10.0, v) for v in vs30s],
    vs30_lbls, c_vs30s, '^',
    'Site Class (Vs30)', 'Mw = 6.5, Rjb = 10 km',
    'parametric_vs30.png'
)


# SHAP Analysis

In [ ]:
import shap

# ── 1. Target outputs: PGA, PGV, Sa(0.10s), Sa(1.00s) ────────────────────
_sel_t  = [0.1, 1.0]
_sa_idx = [min(range(len(periods)), key=lambda i: abs(periods[i] - t)) for t in _sel_t]
sel_out  = [0, 1] + [2 + i for i in _sa_idx]
sel_lbls = ['PGA', 'PGV', f'Sa({periods[_sa_idx[0]]:.2f}s)', f'Sa({periods[_sa_idx[1]]:.2f}s)']

class _SelOut(nn.Module):
    def __init__(self, base, idx): super().__init__(); self.base, self.idx = base, idx
    def forward(self, x): return self.base(x)[:, self.idx]

sel_model = _SelOut(final_model, sel_out).to(device).eval()

# ── 2. Background (1000) + explain (1000) samples ────────────────────────────
rng_s = np.random.default_rng(7)
bg_t  = torch.FloatTensor(X_train_scaled[rng_s.choice(len(X_train_scaled), 1000, replace=False)]).to(device)
ei    = rng_s.choice(len(X_test_scaled), 1000, replace=False)
exp_t = torch.FloatTensor(X_test_scaled[ei]).to(device)
X_exp = X_test_scaled[ei]                   # (1000, 5)
n_s, n_f = X_exp.shape
n_out = len(sel_out)

# ── 3. GradientExplainer (cuDNN disabled for LSTM eval backward) ──────────
print("Computing SHAP values (GradientExplainer, 50 bg × 1000 samples × 4 outputs)…")
_cudnn_prev = torch.backends.cudnn.enabled
torch.backends.cudnn.enabled = False
try:
    explainer = shap.GradientExplainer(sel_model, bg_t)
    raw_shap  = explainer.shap_values(exp_t)
finally:
    torch.backends.cudnn.enabled = _cudnn_prev

# ── 4. Robust parser → list of n_out arrays, each (n_s, n_f) ─────────────
def _to_list(raw, n_out, n_s, n_f):
    """Handle every known SHAP output format → list of n_out × (n_s, n_f) arrays."""
    print(f"  raw_shap type={type(raw).__name__}, ", end="")

    # Explanation object (.values attribute)
    if hasattr(raw, 'values'):
        arr = np.array(raw.values)
    elif isinstance(raw, list):
        # List of n_out arrays each (n_s, n_f)
        if len(raw) == n_out:
            result = [np.array(s[0] if isinstance(s, list) else s) for s in raw]
            if all(r.ndim == 2 and r.shape == (n_s, n_f) for r in result):
                print(f"list of {n_out} arrays, shape={result[0].shape}")
                return result
        # Single-element list wrapping a 3-D array
        arr = np.array(raw[0] if len(raw) == 1 else raw)
    else:
        arr = np.array(raw)

    print(f"array shape={arr.shape}")
    # All known 3-D axis orderings
    if arr.ndim == 3:
        if arr.shape == (n_s, n_out, n_f): return [arr[:, i, :] for i in range(n_out)]
        if arr.shape == (n_out, n_s, n_f): return [arr[i]        for i in range(n_out)]
        if arr.shape == (n_s, n_f, n_out): return [arr[:, :, i]  for i in range(n_out)]
        if arr.shape == (n_f, n_s, n_out): return [arr[:, :, i].T for i in range(n_out)]
    # 2-D single output
    if arr.ndim == 2 and arr.shape == (n_s, n_f):
        return [arr]
    raise ValueError(f"Cannot parse SHAP array of shape {arr.shape} "
                     f"for n_out={n_out}, n_s={n_s}, n_f={n_f}")

shap_vals = _to_list(raw_shap, n_out, n_s, n_f)
assert len(shap_vals) == n_out, f"Expected {n_out} outputs, got {len(shap_vals)}"
assert all(sv.shape == (n_s, n_f) for sv in shap_vals), \
    f"Bad sv shapes: {[sv.shape for sv in shap_vals]}, expected ({n_s},{n_f})"

print(f"✓ SHAP ready: {n_out} outputs × {n_s} samples × {n_f} features")
for lbl, sv in zip(sel_lbls, shap_vals):
    print(f"   {lbl}: shape={sv.shape}")

feat_lbls = ['Mw', '$R_{JB}$ (km)', '$\\log_{10}(R_{JB})$', '$\\log_{10}(V_{S30})$', 'Fault Type']
feat_norm = (X_exp - X_exp.min(0)) / (X_exp.max(0) - X_exp.min(0)).clip(1e-8)


In [ ]:

# ── Re-parse raw_shap with corrected axis-ordering logic ─────────────────
# raw_shap is already computed above; only run this cell if plots errored.
n_out = len(sel_out)   # ensure n_out is defined even if cell 27 wasn't re-run

def _to_list(raw, n_out, n_s, n_f):
    """Handle every known SHAP output format → list of n_out × (n_s, n_f) arrays."""
    print(f"  raw_shap type={type(raw).__name__}", end="")
    if hasattr(raw, 'values'):
        arr = np.array(raw.values)
    elif isinstance(raw, list):
        if len(raw) == n_out:
            result = [np.array(s[0] if isinstance(s, list) else s) for s in raw]
            if all(r.ndim == 2 and r.shape == (n_s, n_f) for r in result):
                print(f"  → list of {n_out} × {result[0].shape}")
                return result
        arr = np.array(raw[0] if len(raw) == 1 else raw)
    else:
        arr = np.array(raw)
    print(f"  → array shape={arr.shape}")
    if arr.ndim == 3:
        if arr.shape == (n_s, n_out, n_f): return [arr[:, i, :] for i in range(n_out)]
        if arr.shape == (n_out, n_s, n_f): return [arr[i]        for i in range(n_out)]
        if arr.shape == (n_s, n_f, n_out): return [arr[:, :, i]  for i in range(n_out)]
        if arr.shape == (n_f, n_s, n_out): return [arr[:, :, i].T for i in range(n_out)]
    if arr.ndim == 2 and arr.shape == (n_s, n_f):
        return [arr]
    raise ValueError(f"Cannot parse shape {arr.shape} for n_out={n_out}, n_s={n_s}, n_f={n_f}")

shap_vals = _to_list(raw_shap, n_out, n_s, n_f)
assert all(sv.shape == (n_s, n_f) for sv in shap_vals), \
    f"Bad shapes: {[sv.shape for sv in shap_vals]}"

feat_lbls = ['Mw', '$R_{JB}$ (km)', '$\\log_{10}(R_{JB})$', '$\\log_{10}(V_{S30})$', 'Fault Type']
feat_norm = (X_exp - X_exp.min(0)) / (X_exp.max(0) - X_exp.min(0)).clip(1e-8)

print(f"\n✓ SHAP parsed: {n_out} outputs × {shap_vals[0].shape[0]} samples × {shap_vals[0].shape[1]} features")
for lbl, sv in zip(sel_lbls, shap_vals):
    print(f"   {lbl}: shape={sv.shape}")


In [ ]:

# ── Per-output figure: beeswarm (left) + importance bar chart (right) ─────
cmap      = plt.cm.RdBu_r
file_tags = ['PGA', 'PGV',
             f'Sa{periods[_sa_idx[0]]:.2f}s'.replace('.', 'pt'),
             f'Sa{periods[_sa_idx[1]]:.2f}s'.replace('.', 'pt')]

for sv, lbl, ftag in zip(shap_vals, sel_lbls, file_tags):
    mean_abs = np.abs(sv).mean(0)          # (n_f,) — one importance value per feature
    sort_idx = np.argsort(mean_abs)        # ascending → bottom-to-top on horizontal bar

    fig, (ax_bee, ax_bar) = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('white')
    fig.suptitle(f'SHAP Analysis — {lbl}', fontsize=14, fontweight='bold', y=1.02)

    # ── Beeswarm ──────────────────────────────────────────────────────────
    rng_j = np.random.default_rng(99)
    for f in range(n_f):
        y_pos = (n_f - 1 - f) + rng_j.uniform(-0.30, 0.30, sv.shape[0])
        ax_bee.scatter(sv[:, f], y_pos,
                       c=feat_norm[:, f], cmap=cmap,
                       s=25, alpha=0.65, edgecolors='none',
                       rasterized=True, vmin=0, vmax=1)
    ax_bee.axvline(0, color='#2C3E50', lw=1.3, linestyle='--', alpha=0.55, zorder=5)
    ax_bee.set_yticks(range(n_f))
    ax_bee.set_yticklabels(feat_lbls[::-1], fontsize=11)
    ax_bee.set_xlabel('SHAP value', fontsize=12, fontweight='bold')
    ax_bee.set_title('Feature Impact Distribution', fontsize=12, fontweight='bold')
    ax_bee.grid(True, axis='x', alpha=0.2, linewidth=0.6)
    ax_bee.set_facecolor('#FAFAFA')
    ax_bee.tick_params(axis='x', labelsize=10)
    for sp in ax_bee.spines.values(): sp.set_linewidth(0.6)

    # Colorbar for beeswarm
    sm   = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
    cbar = fig.colorbar(sm, ax=ax_bee, orientation='vertical',
                        fraction=0.045, pad=0.03, shrink=0.85)
    cbar.set_label('Feature value (low → high)', fontsize=10, fontweight='bold')
    cbar.set_ticks([0, 0.5, 1]); cbar.set_ticklabels(['Low', 'Mid', 'High'], fontsize=9.5)

    # ── Feature importance bar chart ──────────────────────────────────────
    bar_colours = plt.cm.YlOrRd(np.linspace(0.35, 0.85, n_f))
    bars = ax_bar.barh([feat_lbls[i] for i in sort_idx],
                       mean_abs[sort_idx],
                       color=bar_colours, edgecolor='black', linewidth=0.8, height=0.6)
    for bar, val in zip(bars, mean_abs[sort_idx]):
        ax_bar.text(bar.get_width() + mean_abs.max() * 0.02,
                    bar.get_y() + bar.get_height() / 2,
                    f'{val:.4f}', va='center', ha='left',
                    fontsize=10.5, fontweight='bold', color='#2C3E50')
    ax_bar.set_xlabel('Mean |SHAP value|', fontsize=12, fontweight='bold')
    ax_bar.set_title('Feature Importance', fontsize=12, fontweight='bold')
    ax_bar.set_xlim(0, mean_abs.max() * 1.30)
    ax_bar.grid(True, axis='x', alpha=0.25, linewidth=0.6)
    ax_bar.set_facecolor('#FAFAFA')
    ax_bar.tick_params(labelsize=10.5)
    for sp in ax_bar.spines.values(): sp.set_linewidth(0.6)

    plt.tight_layout()
    save_path = f'{OUTPUT_DIR}/shap_{ftag}.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"✓ Saved: {save_path}")
